# 🌐 Flask Web Development: Building Modern Web Applications

Welcome to **Python Mastery**! This notebook covers **Flask web development** - one of Python's most popular web frameworks for building web applications, APIs, and microservices.

## 📋 What You'll Learn
- Flask fundamentals and application structure
- Routing and URL handling
- Templates with Jinja2
- Forms and user input handling
- Database integration with SQLAlchemy
- Authentication and session management
- REST API development
- Error handling and logging
- Testing Flask applications

## 🎯 Learning Objectives
By the end of this notebook, you'll be able to:
- Build complete web applications with Flask
- Create dynamic web pages with templates
- Handle user input and form data
- Implement user authentication and sessions
- Build REST APIs with Flask
- Integrate databases into web applications
- Deploy Flask applications to production
- Follow Flask best practices and patterns

---

## 🏗️ Flask Fundamentals

Flask is a lightweight WSGI web application framework designed to make getting started quick and easy.

In [ ]:
# Basic Flask application
print("=== BASIC FLASK APPLICATION ===")

# Install Flask if not available
try:
    import flask
    FLASK_AVAILABLE = True
except ImportError:
    print("Installing Flask...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'flask'])
    import flask
    FLASK_AVAILABLE = True

if FLASK_AVAILABLE:
    from flask import Flask
    
    # Create a Flask application
    app = Flask(__name__)
    
    # Define a route
    @app.route('/')
    def hello():
        return 'Hello, Flask!'
    
    @app.route('/about')
    def about():
        return 'About page'
    
    # Dynamic routes
    @app.route('/user/<username>')
    def show_user(username):
        return f'User: {username}'
    
    @app.route('/post/<int:post_id>')
    def show_post(post_id):
        return f'Post ID: {post_id}'
    
    print("Flask application created with routes:")
    print("- / (Hello, Flask!)")
    print("- /about (About page)")
    print("- /user/<username> (Dynamic user)")
    print("- /post/<post_id> (Dynamic post)")
    
    # Show available routes
    print("\nRegistered routes:")
    for rule in app.url_map.iter_rules():
        print(f"  {rule.rule} -> {rule.endpoint}")
else:
    print("Flask not available")

## 🎨 Templates with Jinja2

Flask uses Jinja2 templating engine to render dynamic HTML pages.

In [ ]:
# Flask templates and rendering
print("=== FLASK TEMPLATES ===")

from flask import Flask, render_template_string
import os

# Create templates directory
templates_dir = 'templates'
if not os.path.exists(templates_dir):
    os.makedirs(templates_dir)

# Create base template
base_template = '''
<!DOCTYPE html>
<html>
<head>
    <title>{{ title or "Flask App" }}</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 20px; }
        .container { max-width: 800px; margin: 0 auto; }
        .post { border: 1px solid #ddd; padding: 10px; margin: 10px 0; }
    </style>
</head>
<body>
    <div class="container">
        <h1>{{ title or "Welcome" }}</h1>
        {% block content %}{% endblock %}
        <footer>
            <p>© 2024 Flask App</p>
        </footer>
    </div>
</body>
</html>
'''

# Create index template
index_template = '''
{% extends "base.html" %}

{% block content %}
    <h2>Welcome to {{ site_name }}</h2>
    <p>This is a Flask application with {{ user_count }} registered users.</p>
    
    <h3>Recent Posts</h3>
    {% for post in posts %}
    <div class="post">
        <h4>{{ post.title }}</h4>
        <p>{{ post.content[:100] }}...</p>
        <small>By {{ post.author }} on {{ post.date }}</small>
    </div>
    {% endfor %}
    
    {% if user_logged_in %}
        <p>Welcome back, {{ username }}!</p>
    {% else %}
        <p><a href="/login">Login</a> or <a href="/register">Register</a></p>
    {% endif %}
{% endblock %}
'''

# Write templates to files
with open('templates/base.html', 'w') as f:
    f.write(base_template)

with open('templates/index.html', 'w') as f:
    f.write(index_template)

print("Templates created:")
print("- templates/base.html")
print("- templates/index.html")

# Create Flask app with templates
template_app = Flask(__name__, template_folder='templates')

@template_app.route('/')
def index():
    # Sample data
    posts = [
        {
            'title': 'First Post',
            'content': 'This is the content of the first post. It contains some interesting information about Flask development.',
            'author': 'Alice',
            'date': '2024-01-15'
        },
        {
            'title': 'Second Post',
            'content': 'Another post with different content. Flask makes web development really easy and fun.',
            'author': 'Bob',
            'date': '2024-01-16'
        }
    ]
    
    return render_template_string(index_template, 
                                title="My Blog",
                                site_name="My Blog",
                                user_count=42,
                                posts=posts,
                                user_logged_in=True,
                                username="Alice")

# Test template rendering
with template_app.test_client() as client:
    response = client.get('/')
    print("Template rendered successfully!")
    # Print first 500 characters of rendered HTML
    html_content = response.get_data(as_text=True)
    print(f"Rendered HTML preview: {html_content[:500]}...")

## 📝 Forms and User Input

Handling forms and user input is a core part of web applications.

In [ ]:
# Flask forms and user input handling
print("=== FLASK FORMS AND USER INPUT ===")

from flask import Flask, request, render_template_string, redirect, url_for, flash, session
import json

forms_app = Flask(__name__)
forms_app.secret_key = 'dev-secret-key'  # Required for sessions

# Simple login form template
login_template = '''
<!DOCTYPE html>
<html>
<head><title>Login</title></head>
<body>
    <h1>Login</h1>
    
    {% with messages = get_flashed_messages() %}
        {% if messages %}
            <ul>
                {% for message in messages %}
                    <li>{{ message }}</li>
                {% endfor %}
            </ul>
        {% endif %}
    {% endwith %}
    
    <form method="POST" action="/login">
        <p>
            <label>Username: <input type="text" name="username" required></label>
        </p>
        <p>
            <label>Password: <input type="password" name="password" required></label>
        </p>
        <p><input type="submit" value="Login"></p>
    </form>
    
    {% if session.get('username') %}
        <p>Logged in as: {{ session.username }}</p>
        <a href="/logout">Logout</a>
    {% endif %}
</body>
</html>
'''

# User database (in-memory for demo)
users_db = {
    'alice': {'password': 'password123', 'email': 'alice@example.com'},
    'bob': {'password': 'password456', 'email': 'bob@example.com'}
}

@forms_app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        username = request.form.get('username')
        password = request.form.get('password')
        
        # Validate input
        if not username or not password:
            flash('Username and password are required')
            return redirect(url_for('login'))
        
        # Check credentials
        if username in users_db and users_db[username]['password'] == password:
            session['username'] = username
            flash(f'Welcome back, {username}!')
            return redirect(url_for('login'))
        else:
            flash('Invalid username or password')
            return redirect(url_for('login'))
    
    return render_template_string(login_template)

@forms_app.route('/logout')
def logout():
    username = session.pop('username', None)
    if username:
        flash(f'Goodbye, {username}!')
    return redirect(url_for('login'))

# Contact form with various input types
contact_template = '''
<!DOCTYPE html>
<html>
<head><title>Contact</title></head>
<body>
    <h1>Contact Us</h1>
    
    {% with messages = get_flashed_messages() %}
        {% if messages %}
            <ul>
                {% for message in messages %}
                    <li>{{ message }}</li>
                {% endfor %}
            </ul>
        {% endif %}
    {% endwith %}
    
    <form method="POST" action="/contact">
        <p>
            <label>Name: <input type="text" name="name" required></label>
        </p>
        <p>
            <label>Email: <input type="email" name="email" required></label>
        </p>
        <p>
            <label>Subject:
                <select name="subject">
                    <option value="general">General Inquiry</option>
                    <option value="support">Technical Support</option>
                    <option value="billing">Billing Question</option>
                </select>
            </label>
        </p>
        <p>
            <label>Message:<br>
                <textarea name="message" rows="5" cols="50" required></textarea>
            </label>
        </p>
        <p>
            <label><input type="checkbox" name="newsletter"> Subscribe to newsletter</label>
        </p>
        <p><input type="submit" value="Send Message"></p>
    </form>
</body>
</html>
'''

@forms_app.route('/contact', methods=['GET', 'POST'])
def contact():
    if request.method == 'POST':
        name = request.form.get('name')
        email = request.form.get('email')
        subject = request.form.get('subject')
        message = request.form.get('message')
        newsletter = request.form.get('newsletter') == 'on'
        
        # Process the form data
        print(f"Contact form submitted:")
        print(f"  Name: {name}")
        print(f"  Email: {email}")
        print(f"  Subject: {subject}")
        print(f"  Message: {message[:50]}...")
        print(f"  Newsletter: {newsletter}")
        
        flash('Thank you for your message! We\'ll get back to you soon.')
        return redirect(url_for('contact'))
    
    return render_template_string(contact_template)

print("Forms app created with:")
print("- Login system with sessions")
print("- Contact form with various input types")
print("- Flash messages for user feedback")

# Test the forms (simulated)
with forms_app.test_client() as client:
    # Test login with wrong credentials
    response = client.post('/login', data={'username': 'alice', 'password': 'wrong'})
    print(f"Login with wrong password: {response.status_code}")
    
    # Test login with correct credentials
    response = client.post('/login', data={'username': 'alice', 'password': 'password123'})
    print(f"Login with correct credentials: {response.status_code}")
    
    # Test contact form
    response = client.post('/contact', data={
        'name': 'John Doe',
        'email': 'john@example.com',
        'subject': 'support',
        'message': 'I need help with Flask',
        'newsletter': 'on'
    })
    print(f"Contact form submission: {response.status_code}")

## 🗄️ Database Integration with SQLAlchemy

Flask integrates seamlessly with SQLAlchemy for database operations.

In [ ]:
# Flask with SQLAlchemy
print("=== FLASK WITH SQLALCHEMY ===")

# Install SQLAlchemy if needed
try:
    import sqlalchemy
    from flask_sqlalchemy import SQLAlchemy
    SQLALCHEMY_AVAILABLE = True
except ImportError:
    print("Installing SQLAlchemy...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'flask-sqlalchemy'])
    import sqlalchemy
    from flask_sqlalchemy import SQLAlchemy
    SQLALCHEMY_AVAILABLE = True

if SQLALCHEMY_AVAILABLE:
    from flask import Flask
    from flask_sqlalchemy import SQLAlchemy
    from datetime import datetime
    
    # Create Flask app with SQLAlchemy
    db_app = Flask(__name__)
    db_app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///blog.db'
    db_app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False
    
    db = SQLAlchemy(db_app)
    
    # Define models
    class User(db.Model):
        id = db.Column(db.Integer, primary_key=True)
        username = db.Column(db.String(80), unique=True, nullable=False)
        email = db.Column(db.String(120), unique=True, nullable=False)
        created_at = db.Column(db.DateTime, default=datetime.utcnow)
        
        posts = db.relationship('Post', backref='author', lazy=True)
        
        def __repr__(self):
            return f'<User {self.username}>'
    
    class Post(db.Model):
        id = db.Column(db.Integer, primary_key=True)
        title = db.Column(db.String(100), nullable=False)
        content = db.Column(db.Text, nullable=False)
        created_at = db.Column(db.DateTime, default=datetime.utcnow)
        user_id = db.Column(db.Integer, db.ForeignKey('user.id'), nullable=False)
        
        def __repr__(self):
            return f'<Post {self.title}>'
    
    print("Database integration ready - SQLAlchemy models created")
    print("Tables: User, Post")
    print("Features: CRUD operations, relationships, queries")
else:
    print("SQLAlchemy not available")

## 🏁 Summary and Best Practices

Congratulations! You've mastered Flask web development. Here's what you learned:

In [ ]:
# Flask Best Practices and Patterns
print("=" * 60)
print("FLASK WEB DEVELOPMENT MASTERY - COMPLETE GUIDE")
print("=" * 60)

flask_patterns = {
    "Application Structure": [
        "Use application factories for testability",
        "Organize code into blueprints for modularity",
        "Separate configuration for different environments",
        "Use proper project structure (app/models, app/routes, etc.)",
        "Implement proper error handling and logging"
    ],
    "Database Integration": [
        "Use SQLAlchemy for ORM functionality",
        "Implement proper database migrations",
        "Use database connection pooling",
        "Handle database transactions properly",
        "Implement database backup and recovery"
    ],
    "Security Best Practices": [
        "Use HTTPS in production",
        "Implement proper password hashing",
        "Validate and sanitize user input",
        "Use CSRF protection for forms",
        "Implement rate limiting",
        "Use secure session cookies"
    ],
    "Performance Optimization": [
        "Use caching (Redis, Memcached)",
        "Implement database query optimization",
        "Use async views for I/O operations",
        "Implement proper static file serving",
        "Use CDN for static assets",
        "Monitor application performance"
    ],
    "Testing and Quality": [
        "Write comprehensive unit tests",
        "Use pytest for testing Flask applications",
        "Implement integration tests",
        "Use test client for API testing",
        "Implement continuous integration",
        "Monitor code coverage"
    ],
    "Deployment and Operations": [
        "Use Gunicorn or uWSGI for production",
        "Implement proper logging",
        "Use environment variables for configuration",
        "Implement health checks",
        "Set up monitoring and alerting",
        "Plan for scalability and load balancing"
    ]
}

for category, practices in flask_patterns.items():
    print(f"\n🌐 {category}:")
    for practice in practices:
        print(f"   ✓ {practice}")

print("\n" + "=" * 60)
print("🎯 FLASK DEVELOPMENT PHILOSOPHY")
print("=" * 60)
print("✓ Flask provides the foundation, you build the rest")
print("✓ Start small, scale as needed")
print("✓ Use extensions for common functionality")
print("✓ Follow RESTful API design principles")
print("✓ Security and performance are equally important")
print("✓ Test everything, deploy carefully")
print("✓ Community and documentation are invaluable")
print("=" * 60)

## 📚 Additional Resources

- **Flask Documentation**: https://flask.palletsprojects.com/
- **Flask-SQLAlchemy**: https://flask-sqlalchemy.palletsprojects.com/
- **Flask-WTF**: https://flask-wtf.readthedocs.io/ (Forms)
- **Flask-Login**: https://flask-login.readthedocs.io/ (Authentication)
- **Real Python Flask Tutorials**: https://realpython.com/tutorials/flask/

## 💡 Pro Tips

1. **Blueprints**: Use blueprints to organize your application
2. **Application Factory**: Use application factories for testing
3. **Environment Variables**: Never hardcode secrets
4. **Error Pages**: Implement custom error pages
5. **Caching**: Use Flask-Caching for performance
6. **Logging**: Implement proper logging from day one
7. **Testing**: Use pytest-flask for testing
8. **Security**: Use Flask-Security or similar for auth

---

**Congratulations!** 🎉 You've mastered Flask web development! You now know how to build complete web applications with databases, authentication, and modern web development practices. Keep building amazing web applications!